# Machine Learning Zoomcamp 2026 - Homework 6: Trees and Ensemble Learning

This notebook contains the complete solution for Homework 6, predicting car fuel efficiency (`fuel_efficiency_mpg`) using decision trees, random forests, and XGBoost.


## 1. Preparing the Dataset

Guidelines:
* Fill missing values with zeros.
* Do the split with these exact calls:
  ```python
  df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=1)
  df_train, df_val = train_test_split(df_full_train, test_size=0.25, random_state=1)
  ```
* Use `DictVectorizer(sparse=True)` to turn the dataframes into matrices.


In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction import DictVectorizer
from sklearn.tree import DecisionTreeRegressor, export_text
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error
import xgboost as xgb


In [ ]:
df=pd.read_csv('data/car_fuel_efficiency_2026.csv')
df.head()


Dataset shape: (10000, 11)


,model_year,origin,fuel_type,drivetrain,num_doors,engine_displacement,num_cylinders,horsepower,vehicle_weight,acceleration,fuel_efficiency_mpg
0,2006,Europe,Gasoline,Front-wheel drive,4,2180,6,243.0,3870,NaN,31.9
1,2008,Europe,Diesel,Front-wheel drive,4,2390,6,272.0,4210,NaN,31.3
2,1996,Asia,Gasoline,Front-wheel drive,5,2320,6,267.0,4240,17.3,27.5
3,1989,Europe,Gasoline,Front-wheel drive,4,2130,6,258.0,4490,18.7,28.5
4,1994,USA,Diesel,Front-wheel drive,3,2580,7,304.0,4510,17.5,31.0


In [3]:
df.columns=df.columns.str.lower()
df.head()

,model_year,origin,fuel_type,drivetrain,num_doors,engine_displacement,num_cylinders,horsepower,vehicle_weight,acceleration,fuel_efficiency_mpg
0,2006,Europe,Gasoline,Front-wheel drive,4,2180,6,243.0,3870,NaN,31.9
1,2008,Europe,Diesel,Front-wheel drive,4,2390,6,272.0,4210,NaN,31.3
2,1996,Asia,Gasoline,Front-wheel drive,5,2320,6,267.0,4240,17.3,27.5
3,1989,Europe,Gasoline,Front-wheel drive,4,2130,6,258.0,4490,18.7,28.5
4,1994,USA,Diesel,Front-wheel drive,3,2580,7,304.0,4510,17.5,31.0


In [4]:
df.isnull().sum()

model_year               0
origin                   0
fuel_type                0
drivetrain               0
num_doors                0
engine_displacement      0
num_cylinders            0
horsepower             877
vehicle_weight           0
acceleration           264
fuel_efficiency_mpg      0
dtype: int64

In [5]:
df=df.fillna(0)

In [ ]:
# Train, validation, test split
df_full_train,df_test=train_test_split(df,test_size=0.2,random_state=1)
df_train, df_val=train_test_split(
    df_full_train,test_size=0.25,random_state=1
)

df_train=df_train.reset_index(drop=True)
df_val=df_val.reset_index(drop=True)
df_test=df_test.reset_index(drop=True)

y_train=df_train.fuel_efficiency_mpg.values
y_val=df_val.fuel_efficiency_mpg.values
y_test=df_test.fuel_efficiency_mpg.values

del df_train['fuel_efficiency_mpg']
del df_val['fuel_efficiency_mpg']
del df_test['fuel_efficiency_mpg']


In [ ]:
train_dicts=df_train.to_dict(orient='records')
val_dicts=df_val.to_dict(orient='records')
test_dicts=df_test.to_dict(orient='records')

dv=DictVectorizer(sparse=True)
X_train=dv.fit_transform(train_dicts)
X_val=dv.transform(val_dicts)
X_test=dv.transform(test_dicts)

feature_names = dv.get_feature_names_out()
print("featrures:", list(feature_names))


featrures: ['acceleration', 'drivetrain=All-wheel drive', 'drivetrain=Front-wheel drive', 'drivetrain=Rear-wheel drive', 'engine_displacement', 'fuel_type=Diesel', 'fuel_type=Gasoline', 'fuel_type=Hybrid', 'horsepower', 'model_year', 'num_cylinders', 'num_doors', 'origin=Asia', 'origin=Europe', 'origin=USA', 'vehicle_weight']


## Question 1

Let's train a decision tree regressor to predict the `fuel_efficiency_mpg` variable.
* Train a model with `max_depth=1`.

Which original feature is used for splitting the data?
* `'vehicle_weight'`
* `'model_year'`
* `'origin'`
* `'fuel_type'`

**A**: `model_year`


In [ ]:
dt=DecisionTreeRegressor(max_depth=1)
dt.fit(X_train, y_train)

tree_rules=export_text(dt,feature_names=list(feature_names))
print(tree_rules)


|--- model_year <= 1997.50
|   |--- value: [28.61]
|--- model_year >  1997.50
|   |--- value: [31.17]



## Question 2

Train a random forest regressor with these parameters:
* `n_estimators=10`
* `random_state=1`
* `n_jobs=-1` (optional - to make training faster)

What's the RMSE of this model on the validation data?
* 0.1837
* 1.837
* 18.37
* 183.7

**A**: 1,837


In [ ]:
rf=RandomForestRegressor(n_estimators=10,random_state=1,n_jobs=-1)
rf.fit(X_train,y_train)

y_pred=rf.predict(X_val)
rmse_q2=np.sqrt(mean_squared_error(y_val, y_pred))
print(f"RMSE: {rmse_q2:.4f}")


RMSE: 1.8332


## Question 3

Now let's experiment with the `n_estimators` parameter:
* Try these values of this parameter: `[10, 50, 100, 150]`.
* Set `random_state` to `1`.
* Evaluate the model on the validation dataset.

Which value gives the lowest validation RMSE? Consider 3 decimal places for calculating the answer.
* 10
* 50
* 100
* 150

**A**: 150

In [15]:
scores=[]

for n in [10, 50, 100, 150]:
    rf=RandomForestRegressor(n_estimators=n,random_state=1,n_jobs=-1)
    rf.fit(X_train, y_train)
    
    y_pred = rf.predict(X_val)
    rmse = np.sqrt(mean_squared_error(y_val, y_pred))
    scores.append((n, rmse))
    print(f"n_estimators={n:3d} | RMSE = {rmse:.4f} (rounded 3 decimals: {round(rmse, 3)})")


n_estimators= 10 | RMSE = 1.8332 (rounded 3 decimals: 1.833)
n_estimators= 50 | RMSE = 1.7740 (rounded 3 decimals: 1.774)
n_estimators=100 | RMSE = 1.7698 (rounded 3 decimals: 1.77)
n_estimators=150 | RMSE = 1.7692 (rounded 3 decimals: 1.769)


## Question 4

Let's select the best `max_depth`:
* Try different values of `max_depth`: `[10, 15, 20, 25]`
* For each of these values:
  * try different values of `n_estimators` from `[10, 50, 100, 150]`
  * calculate the mean RMSE
* Fix the random seed: `random_state=1`

What's the best `max_depth`, using the mean RMSE?
* 10
* 15
* 20
* 25

**A**: 10

In [16]:
depth_scores = {}

for d in [10, 15, 20, 25]:
    rmses = []
    for n in [10, 50, 100, 150]:
        rf = RandomForestRegressor(max_depth=d, n_estimators=n, random_state=1, n_jobs=-1)
        rf.fit(X_train, y_train)
        
        y_pred = rf.predict(X_val)
        rmse = np.sqrt(mean_squared_error(y_val, y_pred))
        rmses.append(rmse)
    
    mean_rmse = np.mean(rmses)
    depth_scores[d] = mean_rmse
    print(f"max_depth={d:2d} -> Mean RMSE: {mean_rmse:.4f} | Values: {[round(r, 4) for r in rmses]}")

best_depth = min(depth_scores, key=depth_scores.get)
print(f"\nBest max_depth by mean RMSE: {best_depth} (mean RMSE = {depth_scores[best_depth]:.4f})")


max_depth=10 -> Mean RMSE: 1.7544 | Values: [np.float64(1.7823), np.float64(1.7475), np.float64(1.744), np.float64(1.7437)]
max_depth=15 -> Mean RMSE: 1.7842 | Values: [np.float64(1.8314), np.float64(1.7709), np.float64(1.7679), np.float64(1.7668)]
max_depth=20 -> Mean RMSE: 1.7868 | Values: [np.float64(1.8373), np.float64(1.7724), np.float64(1.7689), np.float64(1.7685)]
max_depth=25 -> Mean RMSE: 1.7863 | Values: [np.float64(1.8369), np.float64(1.7721), np.float64(1.7683), np.float64(1.7679)]

Best max_depth by mean RMSE: 10 (mean RMSE = 1.7544)


## Question 5

We can extract feature importance information from tree-based models:
* Train the model with these parameters:
  * `n_estimators=10`,
  * `max_depth=20`,
  * `random_state=1`,
  * `n_jobs=-1` (optional)
* Get the feature importance information from this model

What's the most important feature (among these 4)?
* `vehicle_weight`
* `horsepower`
* `acceleration`
* `engine_displacement`

**A**: `vehicle_weight`


In [ ]:
rf_q5 = RandomForestRegressor(n_estimators=10, max_depth=20, random_state=1, n_jobs=-1)
rf_q5.fit(X_train, y_train)

importances = rf_q5.feature_importances_
df_importance = pd.DataFrame({
    'feature': feature_names,
    'importance': importances
}).sort_values(by='importance', ascending=False).reset_index(drop=True)

target_candidates = ['vehicle_weight', 'horsepower', 'acceleration', 'engine_displacement']
print("\nTarget candidates comparison:")
candidate_df = df_importance[df_importance['feature'].isin(target_candidates)]
print(candidate_df)


All feature importances:
                         feature  importance
0                     model_year    0.296107
1                 vehicle_weight    0.203313
2             fuel_type=Gasoline    0.160712
3                     horsepower    0.078271
4            engine_displacement    0.061281
5                   acceleration    0.054998
6                     origin=USA    0.042152
7   drivetrain=Front-wheel drive    0.020062
8                      num_doors    0.019401
9               fuel_type=Diesel    0.016571
10              fuel_type=Hybrid    0.016272
11    drivetrain=All-wheel drive    0.007951
12                 num_cylinders    0.007318
13                   origin=Asia    0.006515
14   drivetrain=Rear-wheel drive    0.005638
15                 origin=Europe    0.003436

Target candidates comparison:
               feature  importance
1       vehicle_weight    0.203313
3           horsepower    0.078271
4  engine_displacement    0.061281
5         acceleration    0.054998


### Answer for Question 5:
Comparing the four candidates:
1. **`vehicle_weight`**: ~0.2033 (Highest)
2. `horsepower`: ~0.0783
3. `engine_displacement`: ~0.0613
4. `acceleration`: ~0.0550

The most important feature among the choices is **`vehicle_weight`**.


## Question 6

Now let's train an XGBoost model! For this question, we'll tune the `eta` parameter:
* Create DMatrix for train and validation
* Create a watchlist
* Train a model with these parameters for 100 rounds:
  ```python
  xgb_params = {
      'eta': 0.3, 
      'max_depth': 6,
      'min_child_weight': 1,
      'objective': 'reg:squarederror',
      'nthread': 8,
      'seed': 1,
      'verbosity': 1,
  }
  ```
* Now change `eta` from `0.3` to `0.1`.

Which eta leads to the best RMSE score on the validation dataset?
* 0.3
* 0.1
* Both give equal value

**A**: 0,1


In [18]:
features_clean = [f.replace('=', '_').replace('<', '_').replace('>', '_') for f in feature_names]

dtrain = xgb.DMatrix(X_train, label=y_train, feature_names=features_clean)
dval = xgb.DMatrix(X_val, label=y_val, feature_names=features_clean)

watchlist = [(dtrain, 'train'), (dval, 'val')]

for eta in [0.3, 0.1]:
    xgb_params = {
        'eta': eta,
        'max_depth': 6,
        'min_child_weight': 1,
        'objective': 'reg:squarederror',
        'nthread': 8,
        'seed': 1,
        'verbosity': 1,
    }
    
    model = xgb.train(
        xgb_params,
        dtrain,
        num_boost_round=100,
        evals=watchlist,
        verbose_eval=False
    )
    
    y_pred_xgb = model.predict(dval)
    rmse_xgb = np.sqrt(mean_squared_error(y_val, y_pred_xgb))
    print(f"eta = {eta} -> Validation RMSE: {rmse_xgb:.4f}")


eta = 0.3 -> Validation RMSE: 1.8265
eta = 0.1 -> Validation RMSE: 1.7248


### Answer for Question 6:
* `eta = 0.3` -> Validation RMSE: ~1.8265
* `eta = 0.1` -> Validation RMSE: ~1.7248

**`0.1`** gives a significantly lower (better) validation RMSE.
Therefore, the answer is **`0.1`**.
